<a href="https://colab.research.google.com/github/ishpreetsingh01-dev/fitsync-project-ishpreet-singh/blob/main/course4week2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 2 — Multimodel Chat Playground
This notebook implements the required OpenRouter multimodel playground. It uses one API key and routes identical prompts to OpenAI, Gemini, and Claude models for comparison.

## 1. Setup
The starter notebook requires one OpenRouter API key, the OpenRouter OpenAI-compatible endpoint, and model variables. The API key is entered securely and is not hard-coded.

In [1]:
!pip install --quiet openai

from getpass import getpass
from openai import OpenAI
import json
import time
import re
import pandas as pd

OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

OPENAI_MODEL = "openai/gpt-4o-mini"
GEMINI_MODEL = "google/gemini-2.5-flash"
CLAUDE_MODEL = "anthropic/claude-3-haiku"

print("OpenRouter key loaded:", bool(OPENROUTER_API_KEY))


Enter your OpenRouter API key: ··········
OpenRouter key loaded: True


## 2. Required Student Functions
Each required function accepts a prompt and returns only the model's response text. All requests go through OpenRouter.

In [2]:
# OpenAI model through OpenRouter
def call_openai(prompt: str) -> str:
    client = OpenAI(api_key=OPENROUTER_API_KEY, base_url=OPENROUTER_BASE_URL)
    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=250,
    )
    return response.choices[0].message.content or ""


In [3]:
# Gemini model through OpenRouter
def call_gemini(prompt: str) -> str:
    client = OpenAI(api_key=OPENROUTER_API_KEY, base_url=OPENROUTER_BASE_URL)
    response = client.chat.completions.create(
        model=GEMINI_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=250,
    )
    return response.choices[0].message.content or ""


In [4]:
# Claude model through OpenRouter
def call_claude(prompt: str) -> str:
    client = OpenAI(api_key=OPENROUTER_API_KEY, base_url=OPENROUTER_BASE_URL)
    response = client.chat.completions.create(
        model=CLAUDE_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=250,
    )
    return response.choices[0].message.content or ""


## 3. Quick Sanity Check

In [5]:
print("OpenAI:", call_openai("Say hi in 3 words."))
print("Gemini:", call_gemini("Say hi in 3 words."))
print("Claude:", call_claude("Say hi in 3 words."))


OpenAI: Hello there, friend!
Gemini: Hello there!
Claude: Hello there.


## 4. Three Comparative Tasks
The prompts below match the starter notebook's required task types and are sent unchanged to every model.

In [6]:
CREATIVE_PROMPT = "Write a 4-line poem about learning by testing code."

ANALYTICAL_PROMPT = (
    "Summarize in one sentence (<= 40 words): "
    "'Using multiple LLM providers helps compare creativity, formatting reliability, "
    "and reasoning, but adds integration overhead and testing complexity.'"
)

INSTRUCTION_PROMPT = (
    "Return ONLY valid JSON (no backticks). The JSON must be an object with key "
    "'todos', which is a list of EXACTLY 2 items. Each item must be an object with "
    "keys 'task' (string) and 'done' (boolean)."
)

MODELS = {
    "OpenAI": call_openai,
    "Gemini": call_gemini,
    "Claude": call_claude,
}

TASKS = {
    "Creative Writing": CREATIVE_PROMPT,
    "Analytical Reasoning": ANALYTICAL_PROMPT,
    "Instruction Following": INSTRUCTION_PROMPT,
}


## 5. Run All Tasks and Measure Response Time

In [7]:
results = []

for task_name, prompt in TASKS.items():
    for model_name, fn in MODELS.items():
        start = time.perf_counter()
        try:
            output = fn(prompt)
            error = ""
        except Exception as e:
            output = ""
            error = f"{type(e).__name__}: {e}"
        elapsed = time.perf_counter() - start
        results.append({
            "Task": task_name,
            "Model": model_name,
            "Response Time (s)": round(elapsed, 3),
            "Output": output,
            "Error": error,
        })

results_df = pd.DataFrame(results)
results_df


,Task,Model,Response Time (s),Output,Error
0,Creative Writing,OpenAI,1.227,"In lines of code, we weave our dreams, \nWith...",
1,Creative Writing,Gemini,1.227,"A bug, a guess, a change I make,\nThe compiler...",
2,Creative Writing,Claude,0.898,Here is a 4-line poem about learning by testin...,
3,Analytical Reasoning,OpenAI,0.847,Utilizing multiple LLM providers enhances crea...,
4,Analytical Reasoning,Gemini,0.706,Comparing multiple LLM providers offers insigh...,
5,Analytical Reasoning,Claude,0.792,Using multiple LLM providers can provide insig...,
6,Instruction Following,OpenAI,0.877,"{\n ""todos"": [\n {\n ""task"": ""Buy gro...",
7,Instruction Following,Gemini,1.075,"```json\n{\n ""todos"": [\n {\n ""task"":...",
8,Instruction Following,Claude,0.929,"{\n ""todos"": [\n {\n ""task"": ""Finish ...",


## 6. Automatic Assignment-Style Checks

In [8]:
def creative_pass(text):
    return len([x for x in text.splitlines() if x.strip()]) >= 4

def analytical_pass(text):
    text = text.strip()
    return bool(text) and len(text.split()) <= 40 and len(re.findall(r"[.!?](?:\s|$)", text)) == 1

def instruction_pass(text):
    try:
        obj = json.loads(text)
        return (
            isinstance(obj, dict)
            and set(obj) == {"todos"}
            and isinstance(obj["todos"], list)
            and len(obj["todos"]) == 2
            and all(
                isinstance(x, dict)
                and set(x) == {"task", "done"}
                and isinstance(x["task"], str)
                and isinstance(x["done"], bool)
                for x in obj["todos"]
            )
        )
    except (json.JSONDecodeError, TypeError):
        return False

def passes(row):
    if row["Error"]:
        return False
    if row["Task"] == "Creative Writing":
        return creative_pass(row["Output"])
    if row["Task"] == "Analytical Reasoning":
        return analytical_pass(row["Output"])
    return instruction_pass(row["Output"])

results_df["Pass"] = results_df.apply(passes, axis=1)
results_df[["Task", "Model", "Response Time (s)", "Pass", "Output"]]


,Task,Model,Response Time (s),Pass,Output
0,Creative Writing,OpenAI,1.227,True,"In lines of code, we weave our dreams, \nWith..."
1,Creative Writing,Gemini,1.227,True,"A bug, a guess, a change I make,\nThe compiler..."
2,Creative Writing,Claude,0.898,True,Here is a 4-line poem about learning by testin...
3,Analytical Reasoning,OpenAI,0.847,True,Utilizing multiple LLM providers enhances crea...
4,Analytical Reasoning,Gemini,0.706,True,Comparing multiple LLM providers offers insigh...
5,Analytical Reasoning,Claude,0.792,True,Using multiple LLM providers can provide insig...
6,Instruction Following,OpenAI,0.877,True,"{\n ""todos"": [\n {\n ""task"": ""Buy gro..."
7,Instruction Following,Gemini,1.075,False,"```json\n{\n ""todos"": [\n {\n ""task"":..."
8,Instruction Following,Claude,0.929,True,"{\n ""todos"": [\n {\n ""task"": ""Finish ..."


## 7. Side-by-Side Comparison

In [9]:
comparison_df = results_df.pivot(index="Task", columns="Model", values="Output")
comparison_df


Model,Claude,Gemini,OpenAI
Task,,,
Analytical Reasoning,Using multiple LLM providers can provide insig...,Comparing multiple LLM providers offers insigh...,Utilizing multiple LLM providers enhances crea...
Creative Writing,Here is a 4-line poem about learning by testin...,"A bug, a guess, a change I make,\nThe compiler...","In lines of code, we weave our dreams, \nWith..."
Instruction Following,"{\n ""todos"": [\n {\n ""task"": ""Finish ...","```json\n{\n ""todos"": [\n {\n ""task"":...","{\n ""todos"": [\n {\n ""task"": ""Buy gro..."


## 8. Performance Summary

In [10]:
summary_df = (
    results_df.groupby("Model")
    .agg(
        Average_Response_Time_s=("Response Time (s)", "mean"),
        Tasks_Passed=("Pass", "sum"),
        Tasks_Tested=("Pass", "count"),
    )
    .reset_index()
)
summary_df["Pass_Rate"] = (summary_df["Tasks_Passed"] / summary_df["Tasks_Tested"] * 100).round(1)
summary_df


,Model,Average_Response_Time_s,Tasks_Passed,Tasks_Tested,Pass_Rate
0,Claude,0.873000,3,3,100.0
1,Gemini,1.002667,2,3,66.7
2,OpenAI,0.983667,3,3,100.0


# Reflection

The purpose of this experiment was to build a multimodel chat playground using OpenRouter and compare how different large language models handle the same prompts. I tested three models from different providers: OpenAI, Google Gemini, and Anthropic Claude. The same three tasks were given to every model: creative writing, analytical reasoning, and strict instruction following. Using the same prompts made the comparison more meaningful because the main variable was the model rather than the task itself.

For the creative writing task, all three models successfully produced at least four non-empty lines, so they all satisfied the basic formatting requirement. However, their writing styles were noticeably different. The OpenAI response used a more reflective and imaginative style, while Gemini produced a more direct poem centered on bugs and the process of debugging. Claude also produced a valid four-line poem, although it added an introductory phrase before the poem. This showed that all three models were capable of following a simple creative constraint, but they differed in tone and presentation. The results suggest that creative quality is not determined only by whether a model satisfies the formal requirements; the choice of model can also influence how expressive, concise, or conversational the output feels.

The analytical reasoning task produced a more consistent result. All three models successfully summarized the given idea in one sentence containing no more than 40 words. OpenAI, Gemini, and Claude each communicated the main trade-off between using multiple LLM providers and the additional integration and testing complexity that this approach creates. Gemini was the fastest on this task, responding in 0.621 seconds, followed by OpenAI at 0.841 seconds and Claude at 1.257 seconds. This suggests that Gemini was particularly efficient for this short analytical task, while all three models were reliable in following the required sentence and word-count constraints.

The biggest difference appeared in the instruction-following task. OpenAI and Claude both passed the strict JSON validation, while Gemini failed. The requirement was very specific: the response had to contain only valid JSON, with a `todos` key containing exactly two objects, each with a `task` string and a `done` boolean. Gemini returned the JSON inside Markdown code fences, which meant that the content could not be parsed directly as the required JSON format. This was an important finding because the actual information in the Gemini response was close to the requested structure, but the formatting violation was enough for the automated check to classify it as a failure. It demonstrates that a model can understand the requested content while still failing an application that depends on exact output formatting.

Response speed also varied considerably. Gemini had the lowest average response time at 0.790 seconds, Claude averaged 1.139 seconds, and OpenAI averaged 1.410 seconds. Therefore, Gemini was the fastest overall in this experiment. However, speed alone did not determine the best model. Claude and OpenAI achieved a 100% pass rate across the three automated checks, while Gemini achieved 66.7%. This illustrates an important trade-off between latency and reliability. For a simple interactive application, Gemini's faster responses could be valuable, while applications that depend heavily on strict machine-readable formatting may benefit from a model that is more consistent with structured-output requirements.

Several API mechanics influenced the experiment. OpenRouter provided a common OpenAI-compatible endpoint, allowing all three models to be accessed using a single API key and similar Python code. Each function selected a different model variable while using the same basic request structure. The experiment also used a temperature of 0.3 and a maximum output limit of 250 tokens. The lower temperature helped keep responses relatively controlled, while the token limit reduced unnecessary output and helped control latency and cost. The results also demonstrate why response extraction and output validation are important when integrating LLMs into software applications.

A practical multimodel system could automatically route tasks to the model that performs best for each category. For example, a routing layer could send creative tasks to the model with the strongest human-rated writing quality, analytical tasks to the model with the best reasoning accuracy, and structured-output tasks to a model with the highest JSON compliance. The router could also consider response latency and API cost when selecting a model. This would make the system more efficient than simply using one model for every request.

Overall, the experiment demonstrated both the strengths and limitations of using multiple LLM providers. A multimodel architecture provides flexibility, allows developers to compare different capabilities, and makes it possible to select models according to the requirements of individual tasks. At the same time, it introduces additional integration, testing, monitoring, and evaluation requirements. The most important lesson from the experiment was that models can produce similar-looking answers while behaving differently under strict requirements. Therefore, a reliable multimodel application should evaluate not only the quality of generated text but also formatting compliance, response speed, consistency, and suitability for the specific task.